In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

print("All dependencies successfully imported.")

All dependencies successfully imported.


In [2]:
customer_complaints = pd.read_csv("customer_complaints.csv")

In [3]:
customer_complaints

,Complaint ID,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Consumer consent provided?,Submitted via,Date sent to company,Timely response?,Consumer disputed?,Company response to consumer
0,2738619,2017-11-27,Mortgage,Conventional home mortgage,Trouble during payment process,NaN,No,NATIONSTAR MORTGAGE LLC,PA,189XX,Consent provided,Web,2017-11-27,Yes,Unknown,Closed with explanation
1,2933849,2018-06-12,Mortgage,Conventional home mortgage,Trouble during payment process,NaN,No,NATIONSTAR MORTGAGE LLC,CO,80132,Consent provided,Web,2018-06-12,Yes,Unknown,Closed with explanation
2,1165653,2014-12-21,Mortgage,Conventional fixed mortgage,"Loan modification,collection,foreclosure",NaN,No,NATIONSTAR MORTGAGE LLC,CA,91701,Unknown,Web,2014-12-21,Yes,No,Closed with explanation
3,3316943,2019-07-24,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,Yes,Experian Information Solutions Inc.,CA,90221,Consent not provided,Web,2019-07-24,Yes,Unknown,Closed with explanation
4,3157550,2019-02-20,"Payday loan, title loan, or personal loan",Installment loan,Struggling to pay your loan,NaN,No,Santander Consumer USA Holdings Inc.,NC,27410,Consent not provided,Web,2019-02-20,Yes,Unknown,Closed with explanation
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1473402,2374794,2017-03-07,Credit card,NaN,Late fee,NaN,No,JPMORGAN CHASE & CO.,CT,06611,Unknown,Referral,2017-03-13,Yes,No,Closed with monetary relief
1473403,160482,2012-09-27,Credit card,NaN,Advertising and marketing,NaN,No,DISCOVER BANK,TX,75238,Unknown,Web,2012-09-27,Yes,No,Closed with explanation
1473404,1470975,2015-07-16,Credit card,NaN,APR or interest rate,NaN,Yes,BARCLAYS BANK DELAWARE,IL,60417,Consent not provided,Web,2015-07-16,Yes,No,Closed with non-monetary relief
1473405,1250605,2015-02-21,Credit card,NaN,Sale of account,NaN,No,JPMORGAN CHASE & CO.,AL,36695,Unknown,Web,2015-02-21,Yes,Yes,Closed with explanation


In [4]:
def clean_grievance_data(file_path):
    # Load dataset with target tracking fields
    cols = ['Complaint ID', 'Product', 'Sub-product', 'Issue', 'Consumer complaint narrative', 'Company']
    
    # Read the data, fallback to reading everything if specific columns vary
    try:
        df = pd.read_csv("customer_complaints.csv", usecols=cols)
    except Exception:
        df = pd.read_csv("customer_complaints.csv")
    
    # Standardise column naming layout
    df.columns = df.columns.str.replace(' ', '_').str.lower()
    
    # Target narrative column for structural cleaning
    narrative_col = 'consumer_complaint_narrative'
    if narrative_col in df.columns:
        df[narrative_col] = df[narrative_col].fillna('no narrative provided')
    
    # Fill remaining missing parameters globally
    df = df.fillna('unspecified')
    
    # Uniform text normalisation string conversion
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.strip().str.lower()
            
    return df

# Execute preprocessing (Replace with your actual downloaded Kaggle filename)
raw_data_path = "customer_complaints.csv" 
cleaned_df = clean_grievance_data(raw_data_path)

print(f"Data Normalized. Shape: {cleaned_df.shape}")
cleaned_df.head(3)

Data Normalized. Shape: (1473407, 16)


,complaint_id,date_received,product,sub-product,issue,sub-issue,company_public_response,company,state,zip_code,consumer_consent_provided?,submitted_via,date_sent_to_company,timely_response?,consumer_disputed?,company_response_to_consumer
0,2738619,2017-11-27,mortgage,conventional home mortgage,trouble during payment process,unspecified,no,nationstar mortgage llc,pa,189xx,consent provided,web,2017-11-27,yes,unknown,closed with explanation
1,2933849,2018-06-12,mortgage,conventional home mortgage,trouble during payment process,unspecified,no,nationstar mortgage llc,co,80132,consent provided,web,2018-06-12,yes,unknown,closed with explanation
2,1165653,2014-12-21,mortgage,conventional fixed mortgage,"loan modification,collection,foreclosure",unspecified,no,nationstar mortgage llc,ca,91701,unknown,web,2014-12-21,yes,no,closed with explanation


In [ ]:
def classify_priority_triggers(df):
    # Set risk evaluation matrices
    wallet_keywords = ['wallet', 'venmo', 'cashapp', 'digital transaction', 'unauthorized mobile', 'digital wallet']
    atm_keywords = ['atm', 'dispensed', 'cash machine', 'discrepancy', 'overcharged atm', 'automated teller']
    
    # Apply baseline status category
    df['priority_flag'] = 'standard'
    
    # Verify narrative column presence before performing text scans
    if 'consumer_complaint_narrative' in df.columns:
        narrative = df['consumer_complaint_narrative']
        
        is_wallet_fraud = narrative.str.contains('|'.join(wallet_keywords), na=False)
        is_atm_discrepancy = narrative.str.contains('|'.join(atm_keywords), na=False)
        
        df.loc[is_wallet_fraud, 'priority_flag'] = 'high_priority_wallet_fraud'
        df.loc[is_atm_discrepancy, 'priority_flag'] = 'high_priority_atm_discrepancy'
        
    return df

# Transform dataset with structured categorization tags
processed_df = classify_priority_triggers(cleaned_df)

print("Priority classification completed.")
processed_df['priority_flag'].value_counts()

In [ ]:
cleaned_df

In [ ]:
cleaned_df.isnull().sum()

In [ ]:
cleaned_df.head(5)

In [ ]:
cleaned_df.tail(5)

In [ ]:
from sqlalchemy import create_engine, inspect

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Safety Check active again
inspector = inspect(engine)
if not inspector.has_table("complaints"):
    cleaned_df.to_sql(name='complaints', con=engine, if_exists='append', index=False)
    print("Table created and data successfully imported.")
else:
    print("Table already exists in MySQL! Skipping upload to save time.")